### Carga Camada Gold
**🔥 O que esse Etapa faz?**
- ✅ Lê todos arquivos delta da Camada Silver.
- ✅ Para Dimensões: aplicar SCD2 incluindo SK´s.
- ✅ Para Fato: aplicar carga incremental com base no ultimo VendasID
- ✅ Para Fato: salvar arquivos Delta Parquet particionado em Ano/Mes


In [0]:
# Importar bibliotecas
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException
from pyspark.sql.window import Window
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os

# Configuração inicial da SparkSession com configurações otimizadas
spark = SparkSession.builder \
    .appName("Load Data Silver") \
    .config("spark.sql.shuffle.partitions", "200")  \
    .config("spark.sql.files.maxPartitionBytes", "1GB") \
    .config("spark.sql.files.maxRecordsPerFile", "1000000") \
    .config("spark.sql.parquet.compression.codec", "snappy") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

### Função para limpeza de Dataframes

In [0]:
import inspect
from pyspark.sql import DataFrame

def limpeza_dataframes():
    # Obter todas as variáveis locais
    local_vars = inspect.currentframe().f_back.f_locals
    
    # Coletar os nomes dos dataframes em uma lista separada
    df_names = [var_name for var_name, var_value in local_vars.items() if isinstance(var_value, DataFrame)]
    
    # Iterar sobre a lista de nomes de dataframes e eliminá-los
    for var_name in df_names:
        var_value = local_vars[var_name]
        var_value.unpersist()
        del local_vars[var_name]
        print(f"✅ DataFrame '{var_name}' eliminado e deletado!")

###🗒️Configuracão de Parametros e Caminhos

In [0]:
# Define o caminho base do Data Lake
base_path = '/mnt/panex/lhdw'

# Define os caminhos das camadas Silver e Gold
silver_path = f"{base_path}/silver"
gold_path = f"{base_path}/gold/"
tipo_dim ="dimensao"
tipo_fato ="fato"

### 🚀Carga Dim Categoria SCD2

In [0]:
# Definição da tabela
tabela = "categorias"

# Caminhos das camadas Silver e Gold
silver_path_tabela = f"{silver_path}/{tipo_dim}/{tabela}"
gold_path_tabela = f"{gold_path}/{tipo_dim}/{tabela}"

# Inicialize df_novos_registros
df_novos_registros = None

# Leitura da Silver (dados mais recentes)
df_silver = spark.read.format("delta").load(silver_path_tabela)

# Tentar carregar a camada Gold para verificar se existe
try:
    df_gold = spark.read.format("delta").load(gold_path_tabela)
    gold_exists = True
    
    # Obtém o maior sk
    max_sk = df_gold.agg(F.max(f"sk_{tabela}")).collect()[0][0]  
    max_sk = max_sk if max_sk is not None else 0  # Se for None, começa do 0

    # Obtém o tipo da coluna sk na Gold
    sk_type = dict(df_gold.dtypes)[f"sk_{tabela}"]
except AnalysisException:
    gold_exists = False
    max_sk = 0          # Se a Gold não existir, começa do 0
    sk_type = "bigint"  # Default para LongType se a tabela não existir

# Se a Gold não existe, insere todos os registros como novos
if not gold_exists:
    df_silver = (
        df_silver
        .withColumn("data_inicio", F.current_timestamp())
        .withColumn("data_fim", F.lit(None).cast("timestamp"))
        .withColumn("ativo", F.lit(True))
        .withColumn(f"sk_{tabela}", (F.monotonically_increasing_id() + max_sk + 1).cast(LongType()))
    )
    # Salvar na Gold pela primeira vez
    df_silver.write.format("delta").mode("overwrite").save(gold_path_tabela)
    print("✅ Primeira carga concluída!")

else:
    # Criando uma **tabela temporária** no Spark para fazer o merge
    df_silver.createOrReplaceTempView("silver_temp")

    # 🔥 **Passo 1: Atualizar os registros antigos na tabela Gold**
    spark.sql(f"""
        MERGE INTO delta.`{gold_path_tabela}` AS gold
        USING silver_temp AS silver
        ON gold.CategoriaID = silver.CategoriaID AND gold.ativo = True
        WHEN MATCHED AND gold.NomeCategoria <> silver.NomeCategoria THEN
            UPDATE SET 
                gold.data_fim = current_date(),
                gold.ativo = False
    """)

    # 🔥 **Passo 2: Inserir os novos registros na Gold**
    df_novos_registros = spark.sql(f"""
        SELECT 
            silver_temp.CategoriaID, 
            silver_temp.NomeCategoria, 
            current_timestamp() AS data_inicio,
            NULL AS data_fim,
            True AS ativo
        FROM silver_temp
        LEFT JOIN delta.`{gold_path_tabela}` gold
        ON silver_temp.CategoriaID = gold.CategoriaID
        AND gold.ativo = True           -- 🚀 SOMENTE SE NÃO EXISTE UM REGISTRO ATIVO
        WHERE gold.CategoriaID IS NULL  -- 🔥 GARANTE QUE NÃO EXISTA JÁ ATIVO
    """)

    # Criar a surrogate key incremental corretamente
    window_spec = Window.orderBy(F.monotonically_increasing_id())
    df_novos_registros = df_novos_registros.withColumn(
        f"sk_{tabela}",
        (F.row_number().over(window_spec) + max_sk).cast(LongType())  # 🚀 Corrigindo o tipo da coluna
    )

    # Inserir na Gold
    df_novos_registros.write.format("delta").mode("append").save(gold_path_tabela)

    print("✅ Carga SCD2 concluída!")

# Eliminando dataframes da memória
# Chame a função para eliminar todos os dataframes
limpeza_dataframes()

✅ Primeira carga concluída!
✅ DataFrame 'df_silver' eliminado e deletado!


In [0]:
# Leitura da Silver (dados mais recentes)
df = spark.read.format("delta").load("/mnt/panex/lhdw/gold/dimensao/categorias")
display(df)

CategoriaID,NomeCategoria,data_carga,data_inicio,data_fim,ativo,sk_categorias
1,Confections,2025-03-18T22:12:00.124+0000,2025-03-18T22:25:49.142+0000,null,true,1
2,Shell fish,2025-03-18T22:12:00.124+0000,2025-03-18T22:25:49.142+0000,null,true,2
3,Cereals,2025-03-18T22:12:00.124+0000,2025-03-18T22:25:49.142+0000,null,true,3
4,Dairy,2025-03-18T22:12:00.124+0000,2025-03-18T22:25:49.142+0000,null,true,4
5,Beverages,2025-03-18T22:12:00.124+0000,2025-03-18T22:25:49.142+0000,null,true,5
6,Seafood,2025-03-18T22:12:00.124+0000,2025-03-18T22:25:49.142+0000,null,true,6
7,Meat,2025-03-18T22:12:00.124+0000,2025-03-18T22:25:49.142+0000,null,true,7
8,Grain,2025-03-18T22:12:00.124+0000,2025-03-18T22:25:49.142+0000,null,true,8
9,Poultry,2025-03-18T22:12:00.124+0000,2025-03-18T22:25:49.142+0000,null,true,9
10,Snails,2025-03-18T22:12:00.124+0000,2025-03-18T22:25:49.142+0000,null,true,10


### Testando SCD2 na Tabela Categorias

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, lit

dim_categoria_path_silver="/mnt/panex/lhdw/silver/dimensao/categorias"
# Carrega a tabela Delta
dim_categoria_df = spark.read.format("delta").load(dim_categoria_path_silver)

# Cria o objeto DeltaTable
delta_table = DeltaTable.forPath(spark, dim_categoria_path_silver)

# Faz o update na tabela
delta_table.update(
    condition = col("CategoriaID") == 1,
    set = {
        "NomeCategoria": lit("Teste SCD2")
    }
)

# Carrega a tabela Delta
dim_categoria_up = spark.read.format("delta").load(dim_categoria_path_silver)
# Visualiza os resultados atualizados
dim_categoria_up.show()


+-----------+-------------+--------------------+
|CategoriaID|NomeCategoria|          data_carga|
+-----------+-------------+--------------------+
|          1|   Teste SCD2|2025-03-18 22:12:...|
|          2|   Shell fish|2025-03-18 22:12:...|
|          3|      Cereals|2025-03-18 22:12:...|
|          4|        Dairy|2025-03-18 22:12:...|
|          5|    Beverages|2025-03-18 22:12:...|
|          6|      Seafood|2025-03-18 22:12:...|
|          7|         Meat|2025-03-18 22:12:...|
|          8|        Grain|2025-03-18 22:12:...|
|          9|      Poultry|2025-03-18 22:12:...|
|         10|       Snails|2025-03-18 22:12:...|
|         11|      Produce|2025-03-18 22:12:...|
+-----------+-------------+--------------------+



### 🚀Carga Dim Paises SCD2

In [0]:
# Definição da tabela
tabela = "paises"

# Caminhos das camadas Silver e Gold
silver_path_tabela = f"{silver_path}/{tipo_dim}/{tabela}"
gold_path_tabela = f"{gold_path}/{tipo_dim}/{tabela}"

# Leitura da Silver (dados mais recentes)
df_silver = spark.read.format("delta").load(silver_path_tabela)

# Tentar carregar a camada Gold para verificar se existe
try:
    df_gold = spark.read.format("delta").load(gold_path_tabela)
    gold_exists = True
    
    # Obtém o maior sk
    max_sk = df_gold.agg(F.max(f"sk_{tabela}")).collect()[0][0]  
    max_sk = max_sk if max_sk is not None else 0  # Se for None, começa do 0

    # Obtém o tipo da coluna sk na Gold
    sk_type = dict(df_gold.dtypes)[f"sk_{tabela}"]
except AnalysisException:
    gold_exists = False
    max_sk = 0  # Se a Gold não existir, começa do 0
    sk_type = "bigint"  # Default para LongType se a tabela não existir

# Se a Gold não existe, insere todos os registros como novos
if not gold_exists:
    df_silver = (
        df_silver
        .withColumn("data_inicio", F.current_timestamp())
        .withColumn("data_fim", F.lit(None).cast("timestamp"))
        .withColumn("ativo", F.lit(True))
        .withColumn(f"sk_{tabela}", (F.monotonically_increasing_id() + max_sk + 1).cast(LongType()))
    )
    # Salvar na Gold pela primeira vez
    df_silver.write.format("delta").mode("overwrite").save(gold_path_tabela)
    print("✅ Primeira carga concluída!")

else:
    # Criando uma **tabela temporária** no Spark para fazer o merge
    df_silver.createOrReplaceTempView("silver_temp")

    # 🔥 **Passo 1: Atualizar os registros antigos na tabela Gold**
    spark.sql(f"""
        MERGE INTO delta.`{gold_path_tabela}` AS gold
        USING silver_temp AS silver
        ON gold.PaisID = silver.PaisID AND gold.ativo = True
        WHEN MATCHED AND gold.PaisNome <> silver.PaisNome THEN
            UPDATE SET 
                gold.data_fim = current_date(),
                gold.ativo = False
    """)

    # 🔥 **Passo 2: Inserir os novos registros na Gold**
    df_novos_registros = spark.sql(f"""
        SELECT 
            silver_temp.PaisID, 
            silver_temp.PaisNome, 
            current_timestamp() AS data_inicio,
            NULL AS data_fim,
            True AS ativo
        FROM silver_temp
        LEFT JOIN delta.`{gold_path_tabela}` gold
        ON silver_temp.PaisID = gold.PaisID
        AND gold.ativo = True  -- 🚀 SOMENTE SE NÃO EXISTE UM REGISTRO ATIVO
        WHERE gold.PaisID IS NULL  -- 🔥 GARANTE QUE NÃO EXISTA JÁ ATIVO
    """)

    # Criar a surrogate key incremental corretamente
    window_spec = Window.orderBy(F.monotonically_increasing_id())
    df_novos_registros = df_novos_registros.withColumn(
        f"sk_{tabela}",
        (F.row_number().over(window_spec) + max_sk).cast(LongType())  # 🚀 Corrigindo o tipo da coluna
    )

    # Inserir na Gold
    df_novos_registros.write.format("delta").mode("append").save(gold_path_tabela)

    print("✅ Carga SCD2 concluída!")


# Eliminando dataframes da memória
# Chame a função para eliminar todos os dataframes
limpeza_dataframes()

✅ Primeira carga concluída!
✅ DataFrame 'df' eliminado e deletado!
✅ DataFrame 'dim_categoria_df' eliminado e deletado!
✅ DataFrame 'dim_categoria_up' eliminado e deletado!
✅ DataFrame 'df_silver' eliminado e deletado!


### 🚀Carga Dim Vendedores SCD2

In [0]:
# Definição da tabela
tabela = "vendedores"

# Caminhos das camadas Silver e Gold
silver_path_tabela = f"{silver_path}/{tipo_dim}/{tabela}"
gold_path_tabela = f"{gold_path}/{tipo_dim}/{tabela}"

# Leitura da Silver (dados mais recentes)
df_silver = spark.read.format("delta").load(silver_path_tabela)

# Tentar carregar a camada Gold para verificar se existe
try:
    df_gold = spark.read.format("delta").load(gold_path_tabela)
    gold_exists = True
    
    # Obtém o maior sk
    max_sk = df_gold.agg(F.max(f"sk_{tabela}")).collect()[0][0]  
    max_sk = max_sk if max_sk is not None else 0  # Se for None, começa do 0

    # Obtém o tipo da coluna sk na Gold
    sk_type = dict(df_gold.dtypes)[f"sk_{tabela}"]
except AnalysisException:
    gold_exists = False
    max_sk = 0  # Se a Gold não existir, começa do 0
    sk_type = "bigint"  # Default para LongType se a tabela não existir

# Se a Gold não existe, insere todos os registros como novos
if not gold_exists:
    df_silver = (
        df_silver
        .withColumn("data_inicio", F.current_timestamp())
        .withColumn("data_fim", F.lit(None).cast("timestamp"))
        .withColumn("ativo", F.lit(True))
        .withColumn(f"sk_{tabela}", (F.monotonically_increasing_id() + max_sk + 1).cast(LongType()))
    )
    # Salvar na Gold pela primeira vez
    df_silver.write.format("delta").mode("overwrite").save(gold_path_tabela)
    print("✅ Primeira carga concluída!")

else:
    # Criando uma **tabela temporária** no Spark para fazer o merge
    df_silver.createOrReplaceTempView("silver_temp")

    # 🔥 **Passo 1: Atualizar os registros antigos na tabela Gold**
    spark.sql(f"""
        MERGE INTO delta.`{gold_path_tabela}` AS gold
        USING silver_temp AS silver
        ON gold.VendedorID = silver.VendedorID AND gold.ativo = True
        WHEN MATCHED AND gold.Nome <> silver.Nome THEN
            UPDATE SET 
                gold.data_fim = current_date(),
                gold.ativo = False
    """)

    # 🔥 **Passo 2: Inserir os novos registros na Gold**
    df_novos_registros = spark.sql(f"""
        SELECT 
            silver_temp.VendedorID, 
            silver_temp.Nome, 
            current_timestamp() AS data_inicio,
            NULL AS data_fim,
            True AS ativo
        FROM silver_temp
        LEFT JOIN delta.`{gold_path_tabela}` gold
        ON silver_temp.VendedorID = gold.VendedorID
        AND gold.ativo = True  -- 🚀 SOMENTE SE NÃO EXISTE UM REGISTRO ATIVO
        WHERE gold.VendedorID IS NULL  -- 🔥 GARANTE QUE NÃO EXISTA JÁ ATIVO
    """)

    # Criar a surrogate key incremental corretamente
    window_spec = Window.orderBy(F.monotonically_increasing_id())
    df_novos_registros = df_novos_registros.withColumn(
        f"sk_{tabela}",
        (F.row_number().over(window_spec) + max_sk).cast(LongType())  # 🚀 Corrigindo o tipo da coluna
    )

    # Inserir na Gold
    df_novos_registros.write.format("delta").mode("append").save(gold_path_tabela)

    print("✅ Carga SCD2 concluída!")


# Eliminando dataframes da memória
# Chame a função para eliminar todos os dataframes
limpeza_dataframes()

✅ Primeira carga concluída!
✅ DataFrame 'df_silver' eliminado e deletado!


### 🚀Carga Dim Cidades SCD2

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException
from pyspark.sql.types import LongType

# Definição da tabela
tabela = "cidades"
tabela_dependencia = "paises" 

# Caminhos das camadas Silver e Gold
silver_path_tabela = f"{silver_path}/{tipo_dim}/{tabela}"
gold_path_tabela = f"{gold_path}/{tipo_dim}/{tabela}"
gold_path_tabela_dep = f"{gold_path}/{tipo_dim}/{tabela_dependencia}"

# Leitura da Silver (dados mais recentes)
df_silver = spark.read.format("delta").load(silver_path_tabela)

# Leitura da tabela Gold para obter a sk
df_gold_dep = spark.read.format("delta").load(gold_path_tabela_dep)

# Criando a chave  (sk) na tabela
df_silver = df_silver.alias("c")
df_gold_dep = df_gold_dep.alias("p")

df_silver = df_silver.join(
    df_gold_dep,
    df_silver["PaisID"] == df_gold_dep["PaisID"],
    "left"
).select(
    F.col("c.CidadeID"),
    F.col("c.NomeCidade"),
    F.col("c.Cep"),
    F.col(f"p.sk_{tabela_dependencia}").alias(f"sk_{tabela_dependencia}")  # Atribui a chave substituta 
)

# Tentar carregar a camada Gold para verificar se existe
try:
    df_gold = spark.read.format("delta").load(gold_path_tabela)
    gold_exists = True
    
    # Obtém o maior sk_cidades
    max_sk = df_gold.agg(F.max(f"sk_{tabela}")).collect()[0][0]  
    max_sk = max_sk if max_sk is not None else 0  # Se for None, começa do 0

    # Obtém o tipo da coluna sk_cidades na Gold
    sk_type = dict(df_gold.dtypes)[f"sk_{tabela}"]
except AnalysisException:
    gold_exists = False
    max_sk = 0  # Se a Gold não existir, começa do 0
    sk_type = "bigint"  # Default para LongType se a tabela não existir

# Se a Gold não existe, insere todos os registros como novos
if not gold_exists:
    df_silver = (
        df_silver
        .withColumn("data_inicio", F.current_timestamp())
        .withColumn("data_fim", F.lit(None).cast("timestamp"))
        .withColumn("ativo", F.lit(True))
        .withColumn(f"sk_{tabela}", (F.monotonically_increasing_id() + max_sk + 1).cast(LongType()))
    )
    # Salvar na Gold pela primeira vez
    df_silver.write.format("delta").mode("overwrite").save(gold_path_tabela)
    print("✅ Primeira carga concluída!")

else:
    # Criando uma **tabela temporária** no Spark para fazer o merge
    df_silver.createOrReplaceTempView("silver_temp")

    # 🔥 **Passo 1: Atualizar os registros antigos na tabela Gold**
    spark.sql(f"""
        MERGE INTO delta.`{gold_path_tabela}` AS gold
        USING silver_temp AS silver
        ON gold.CidadeID = silver.CidadeID AND gold.ativo = True
        WHEN MATCHED AND (gold.NomeCidade <> silver.NomeCidade OR gold.Cep <> silver.Cep) THEN
            UPDATE SET 
                gold.data_fim = current_date(),
                gold.ativo = False
    """)

    # 🔥 **Passo 2: Inserir os novos registros na Gold**
    df_novos_registros = spark.sql(f"""
        SELECT 
            silver_temp.CidadeID, 
            silver_temp.NomeCidade, 
            silver_temp.Cep,
            silver_temp.sk_{tabela_dependencia},  -- 🚀 Adicionando a chave substituta
            current_timestamp() AS data_inicio,
            NULL AS data_fim,
            True AS ativo
        FROM silver_temp
        LEFT JOIN delta.`{gold_path_tabela}` gold
        ON silver_temp.CidadeID = gold.CidadeID
        AND gold.ativo = True  -- 🚀 SOMENTE SE NÃO EXISTE UM REGISTRO ATIVO
        WHERE gold.CidadeID IS NULL  -- 🔥 GARANTE QUE NÃO EXISTA JÁ ATIVO
    """)

    # Criar a surrogate key incremental corretamente
    window_spec = Window.orderBy(F.monotonically_increasing_id())
    df_novos_registros = df_novos_registros.withColumn(
        f"sk_{tabela}",
        (F.row_number().over(window_spec) + max_sk).cast(LongType())  # 🚀 Corrigindo o tipo da coluna
    )

    # Inserir na Gold
    df_novos_registros.write.format("delta").mode("append").save(gold_path_tabela)

    print("✅ Carga SCD2 concluída!")

# Eliminando dataframes da memória
# Chame a função para eliminar todos os dataframes
limpeza_dataframes()

✅ Primeira carga concluída!
✅ DataFrame 'df_silver' eliminado e deletado!
✅ DataFrame 'df_gold_dep' eliminado e deletado!


### 🚀Carga Dim Produtos SCD2

In [0]:
# Definição da tabela
tabela = "produtos"
tabela_dependencia = "categorias" 

# Caminhos das camadas Silver e Gold
silver_path_tabela = f"{silver_path}/{tipo_dim}/{tabela}"
gold_path_tabela = f"{gold_path}/{tipo_dim}/{tabela}"
gold_path_tabela_dep = f"{gold_path}/{tipo_dim}/{tabela_dependencia}"

# Leitura da Silver (dados mais recentes)
df_silver = spark.read.format("delta").load(silver_path_tabela)

# Leitura da tabela Gold para obter a sk
df_gold_dep = spark.read.format("delta").load(gold_path_tabela_dep)

# Filtrar a tabela de dependências para registros ativos
df_gold_dep = df_gold_dep.filter("ativo = True")

# Criando a chave estrangeira (sk_pais) na tabela cidades
df_silver = df_silver.alias("c")
df_gold_dep = df_gold_dep.alias("p")

df_silver = df_silver.join(
    df_gold_dep,
    df_silver["CategoriaID"] == df_gold_dep["CategoriaID"],
    "left"
).select(
    F.col("c.ProdutoID"),
    F.col("c.ProdutoNome"),
    F.col("c.Preco"),
    F.col("c.Classe"),
    F.col("c.DataCadastro"),
    F.col("c.Resistencia"),
    F.col("c.EAlergico"),
    F.col("c.ValidadeDias"),
    F.col(f"p.sk_{tabela_dependencia}").alias(f"sk_{tabela_dependencia}")  # Atribui a chave substituta
)

# Tentar carregar a camada Gold para verificar se existe
try:
    df_gold = spark.read.format("delta").load(gold_path_tabela)
    gold_exists = True
    
    # Obtém o maior sk
    max_sk = df_gold.agg(F.max(f"sk_{tabela}")).collect()[0][0]  
    max_sk = max_sk if max_sk is not None else 0  # Se for None, começa do 0

    # Obtém o tipo da coluna sk na Gold
    sk_type = dict(df_gold.dtypes)[f"sk_{tabela}"]
except AnalysisException:
    gold_exists = False
    max_sk = 0  # Se a Gold não existir, começa do 0
    sk_type = "bigint"  # Default para LongType se a tabela não existir

# Se a Gold não existe, insere todos os registros como novos
if not gold_exists:
    df_silver = (
        df_silver
        .withColumn("data_inicio", F.current_timestamp())
        .withColumn("data_fim", F.lit(None).cast("timestamp"))
        .withColumn("ativo", F.lit(True))
        .withColumn(f"sk_{tabela}", (F.monotonically_increasing_id() + max_sk + 1).cast(LongType()))
    )
    # Salvar na Gold pela primeira vez
    df_silver.write.format("delta").mode("overwrite").save(gold_path_tabela)
    print("✅ Primeira carga concluída!")

else:
    # Criando uma **tabela temporária** no Spark para fazer o merge
    df_silver.createOrReplaceTempView("silver_temp")

    # 🔥 **Passo 1: Atualizar os registros antigos na tabela Gold**
    spark.sql(f"""
        MERGE INTO delta.`{gold_path_tabela}` AS gold
        USING silver_temp AS silver
        ON gold.ProdutoID = silver.ProdutoID AND gold.ativo = True
        WHEN MATCHED AND (gold.ProdutoNome <> silver.ProdutoNome)  THEN
            UPDATE SET 
                gold.data_fim = current_timestamp(),
                gold.ativo = False
    """)

    # 🔥 **Passo 2: Inserir os novos registros na Gold**
    df_novos_registros = spark.sql(f"""
        SELECT 
            silver_temp.ProdutoID, 
            silver_temp.ProdutoNome, 
            silver_temp.Preco,
            silver_temp.Classe,
            silver_temp.DataCadastro,
            silver_temp.Resistencia,
            silver_temp.EAlergico,
            silver_temp.ValidadeDias,
            silver_temp.sk_{tabela_dependencia},  -- 🚀 Adicionando a chave substituta
            current_timestamp() AS data_inicio,
            NULL AS data_fim,
            True AS ativo
        FROM silver_temp
        LEFT JOIN delta.`{gold_path_tabela}` gold
        ON silver_temp.ProdutoID = gold.ProdutoID
        AND gold.ativo = True  -- 🚀 SOMENTE SE NÃO EXISTE UM REGISTRO ATIVO
        WHERE gold.ProdutoID IS NULL  -- 🔥 GARANTE QUE NÃO EXISTA JÁ ATIVO
    """)

    # Criar a surrogate key incremental corretamente
    window_spec = Window.orderBy(F.monotonically_increasing_id())
    df_novos_registros = df_novos_registros.withColumn(
        f"sk_{tabela}",
        (F.row_number().over(window_spec) + max_sk).cast(LongType())  # 🚀 Corrigindo o tipo da coluna
    )

    # Inserir na Gold
    df_novos_registros.write.format("delta").mode("append").save(gold_path_tabela)

    print("✅ Carga SCD2 concluída!")

# Eliminando dataframes da memória
# Chame a função para eliminar todos os dataframes
limpeza_dataframes()

✅ Primeira carga concluída!
✅ DataFrame 'df_silver' eliminado e deletado!
✅ DataFrame 'df_gold_dep' eliminado e deletado!


### 🚀Carga Dim Cliente SCD2

In [0]:
# Definição da tabela
tabela = "clientes"
tabela_dependencia = "cidades" 

# Caminhos das camadas Silver e Gold
silver_path_tabela = f"{silver_path}/{tipo_dim}/{tabela}"
gold_path_tabela = f"{gold_path}/{tipo_dim}/{tabela}"
gold_path_tabela_dep = f"{gold_path}/{tipo_dim}/{tabela_dependencia}"

# Leitura da Silver (dados mais recentes)
df_silver = spark.read.format("delta").load(silver_path_tabela)

# Leitura da tabela Gold para obter a sk
df_gold_dep = spark.read.format("delta").load(gold_path_tabela_dep)

# Filtrar a tabela de dependências para registros ativos
df_gold_dep = df_gold_dep.filter("ativo = True")

# Criando a chave estrangeira sk na tabela
df_silver = df_silver.alias("c")
df_gold_dep = df_gold_dep.alias("p")

df_silver = df_silver.join(
    df_gold_dep,
    df_silver["CidadeID"] == df_gold_dep["CidadeID"],
    "left"
).select(
    F.col("c.ClienteID"),
    F.col("c.Nome"),
    F.col("c.Endereco"),
    F.col(f"sk_{tabela_dependencia}").alias(f"sk_{tabela_dependencia}")  # Atribui a chave substituta da tabela
)

# Tentar carregar a camada Gold para verificar se existe
try:
    df_gold = spark.read.format("delta").load(gold_path_tabela)
    gold_exists = True
    
    # Obtém o maior sk
    max_sk = df_gold.agg(F.max(f"sk_{tabela}")).collect()[0][0]  
    max_sk = max_sk if max_sk is not None else 0  # Se for None, começa do 0

    # Obtém o tipo da coluna sks na Gold
    sk_type = dict(df_gold.dtypes)[f"sk_{tabela}"]
except AnalysisException:
    gold_exists = False
    max_sk = 0  # Se a Gold não existir, começa do 0
    sk_type = "bigint"  # Default para LongType se a tabela não existir

# Se a Gold não existe, insere todos os registros como novos
if not gold_exists:
    df_silver = (
        df_silver
        .withColumn("data_inicio", F.current_timestamp())
        .withColumn("data_fim", F.lit(None).cast("timestamp"))
        .withColumn("ativo", F.lit(True))
        .withColumn(f"sk_{tabela}", (F.monotonically_increasing_id() + max_sk + 1).cast(LongType()))
    )
    # Salvar na Gold pela primeira vez
    df_silver.write.format("delta").mode("overwrite").save(gold_path_tabela)
    print("✅ Primeira carga concluída!")

else:
    # Criando uma **tabela temporária** no Spark para fazer o merge
    df_silver.createOrReplaceTempView("silver_temp")

    # 🔥 **Passo 1: Atualizar os registros antigos na tabela Gold**
    spark.sql(f"""
        MERGE INTO delta.`{gold_path_tabela}` AS gold
        USING silver_temp AS silver
        ON gold.ClienteID = silver.ClienteID AND gold.ativo = True
        WHEN MATCHED AND (gold.Nome <> silver.Nome) or (gold.Endereco <> silver.Endereco) THEN
            UPDATE SET 
                gold.data_fim = current_timestamp(),
                gold.ativo = False
    """)

    # 🔥 **Passo 2: Inserir os novos registros na Gold**
    df_novos_registros = spark.sql(f"""
        SELECT 
            silver_temp.ClienteID, 
            silver_temp.Nome, 
            silver_temp.Endereco,
            silver_temp.sk_{tabela_dependencia},  -- 🚀 Adicionando a chave substituta
            current_timestamp() AS data_inicio,
            NULL AS data_fim,
            True AS ativo
        FROM silver_temp
        LEFT JOIN delta.`{gold_path_tabela}` gold
        ON silver_temp.ClienteID = gold.ClienteID
        AND gold.ativo = True  -- 🚀 SOMENTE SE NÃO EXISTE UM REGISTRO ATIVO
        WHERE gold.ClienteID IS NULL  -- 🔥 GARANTE QUE NÃO EXISTA JÁ ATIVO
    """)

    # Criar a surrogate key incremental corretamente
    window_spec = Window.orderBy(F.monotonically_increasing_id())
    df_novos_registros = df_novos_registros.withColumn(
        f"sk_{tabela}",
        (F.row_number().over(window_spec) + max_sk).cast(LongType())  # 🚀 Corrigindo o tipo da coluna
    )

    # Inserir na Gold
    df_novos_registros.write.format("delta").mode("append").save(gold_path_tabela)

    print("✅ Carga SCD2 concluída!")

# Eliminando dataframes da memória
# Chame a função para eliminar todos os dataframes
limpeza_dataframes()

✅ Primeira carga concluída!
✅ DataFrame 'df_silver' eliminado e deletado!
✅ DataFrame 'df_gold_dep' eliminado e deletado!


### Verifique as tabelas

In [0]:
# Definição da tabela
tabela = "produtos"
tabela_dim = "dimensao" 

# Caminhos das camadas Silver e Gold
gold_path_tabela = f"{gold_path}/{tipo_dim}/{tabela}"
# Leitura da tabela Gold
df_gold = spark.read.format("delta").load(gold_path_tabela)
display(df_gold.count())
display(df_gold)

# Chame a função para eliminar todos os dataframes
limpeza_dataframes()

452

ProdutoID,ProdutoNome,Preco,Classe,DataCadastro,Resistencia,EAlergico,ValidadeDias,sk_categorias,data_inicio,data_fim,ativo,sk_produtos
1,Flour - Whole Wheat,74.2988,Medium,2018-02-16T08:21:49.190+0000,Durable,Unknown,0.0,3,2025-03-17T22:02:24.646+0000,null,true,1
2,Cookie Chocolate Chip With,91.2329,Medium,2017-02-12T11:39:10.970+0000,Unknown,Unknown,0.0,3,2025-03-17T22:02:24.646+0000,null,true,2
3,Onions - Cippolini,9.1379,Medium,2018-03-15T08:11:51.560+0000,Weak,False,111.0,9,2025-03-17T22:02:24.646+0000,null,true,3
4,"Sauce - Gravy, Au Jus, Mix",54.3055,Medium,2017-07-16T00:46:28.880+0000,Durable,Unknown,0.0,9,2025-03-17T22:02:24.646+0000,null,true,4
5,Artichokes - Jerusalem,65.4771,Low,2017-08-16T14:13:35.430+0000,Durable,True,27.0,2,2025-03-17T22:02:24.646+0000,null,true,5
6,Wine - Magnotta - Cab Sauv,79.7184,High,2017-05-25T15:08:39.690+0000,Unknown,Unknown,0.0,8,2025-03-17T22:02:24.646+0000,null,true,6
7,Table Cloth - 53x69 Colour,31.837,Medium,2017-02-24T15:14:30.050+0000,Durable,False,0.0,9,2025-03-17T22:02:24.646+0000,null,true,7
8,Halibut - Steaks,89.8573,Medium,2018-03-24T05:21:21.890+0000,Unknown,True,108.0,5,2025-03-17T22:02:24.646+0000,null,true,8
9,Rabbit - Whole,84.4219,Medium,2017-06-17T12:12:04.670+0000,Durable,Unknown,0.0,11,2025-03-17T22:02:24.646+0000,null,true,9
10,Scampi Tail,95.0957,Low,2017-07-30T10:11:45.990+0000,Weak,True,105.0,4,2025-03-17T22:02:24.646+0000,null,true,10


✅ DataFrame 'df_gold' eliminado e deletado!


### 🏁✅Load Tabela Fato
- ✔️Carga Incremental com base no ultimo VendasID
- ✔️Trazer SK´s das tabelas dependentes

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

# Definir os caminhos base
base_path = '/mnt/panex/lhdw'
silver_path = f"{base_path}/silver"
gold_path = f"{base_path}/gold"

# Carregar os dataframes das dimensões da camada Gold
df_gold_cliente = spark.read.format("delta").load(f"{gold_path}/{tipo_dim}/clientes")
df_gold_produtos = spark.read.format("delta").load(f"{gold_path}/{tipo_dim}/produtos")
df_gold_vendedores = spark.read.format("delta").load(f"{gold_path}/{tipo_dim}/vendedores")

# Carregar o dataframe fato da camada Silver
df_silver_fato = spark.read.format("delta").load(f"{silver_path}/fato")

# Realizar os joins em uma única operação
df_fato_final = df_silver_fato.alias('fato') \
    .join(
        F.broadcast(df_gold_produtos.alias('produtos')),
        on = [
            F.col('fato.ProdutoID') == F.col('produtos.ProdutoID'),
            F.col('produtos.data_fim').isNull(),
            F.col('produtos.ativo') == True,
        ],
        how='left'
    ).join(
        F.broadcast(df_gold_cliente.alias('clientes')),
        on = [
            F.col('fato.ClienteID') == F.col('clientes.ClienteID'),
            F.col('clientes.data_fim').isNull(),
            F.col('clientes.ativo') == True,
        ],
        how='left'
    ).join(
        F.broadcast(df_gold_vendedores.alias('vendedores')),
        on = [
            F.col('fato.VendedorID') == F.col('vendedores.VendedorID'),
            F.col('vendedores.data_fim').isNull(),
            F.col('vendedores.ativo') == True,
        ],
        how='left'
    ).select(
        F.col('fato.VendasID'),
        F.col('produtos.sk_produtos'),
        F.col('clientes.sk_clientes'),
        F.col('vendedores.sk_vendedores'),
        F.col('fato.Quantidade'),
        F.col('fato.Desconto'),
        F.col('fato.PrecoTotal'),
        F.col('fato.DataVenda'),
        F.col('fato.PrecoUnitario'),
        F.col('fato.Ano'),
        F.col('fato.Mes'),
        F.col('fato.data_carga')
    )

fato_table_path = f"{gold_path}/fato/"

# Verificar se a tabela Gold já existe
try:
    df_gold_fato = spark.read.format("delta").load(fato_table_path)
    gold_exists = True
except AnalysisException:
    gold_exists = False

if gold_exists:
    # Otimizar a leitura filtrando apenas o último mês e ano disponíveis
    max_ano_mes = df_gold_fato.agg(
        F.max("Ano").alias("max_ano"),
        F.max("Mes").alias("max_mes")
    ).collect()[0]
    
    max_ano, max_mes = max_ano_mes["max_ano"], max_ano_mes["max_mes"]
    
    df_gold_fato_filtered = df_gold_fato.filter(
        (F.col("Ano") == max_ano) & (F.col("Mes") == max_mes)
    )
    
    # Obter o valor máximo de VendasID na tabela Gold filtrada
    max_vendas_id = df_gold_fato_filtered.agg(F.max("VendasID")).collect()[0][0]
    
    # Filtrar o df_fato_final para incluir apenas registros com VendasID maior que max_vendas_id
    df_fato_final_to_append = df_fato_final.filter(F.col("VendasID") > max_vendas_id)
    
    # Verificar se há registros novos para inserir
    if df_fato_final_to_append.count() > 0:
        # Fazer o append dos novos registros com partição por Ano e Mes
        df_fato_final_to_append.write.format("delta") \
            .mode("append") \
            .partitionBy("Ano", "Mes") \
            .save(fato_table_path)
        print("✅ Novos registros inseridos na tabela fato!")
    else:
        print("ℹ️ Nenhum novo registro para inserir.")
else:
    # Escrever o dataframe completo na Gold (primeira carga) com partição por Ano e Mes
    df_fato_final.write.format("delta") \
        .mode("overwrite") \
        .partitionBy("Ano", "Mes") \
        .save(fato_table_path)
    print("✅ Tabela fato criada e salva com sucesso!")

# Contando dados na tabela fato
df_fato_contagem = spark.read.format("delta").load(fato_table_path)
print(f"Total de registros na tabela fato: {df_fato_contagem.count()}")

# Chame a função para eliminar todos os dataframes
limpeza_dataframes()


✅ Tabela fato criada e salva com sucesso!
Total de registros na tabela fato: 965446
✅ DataFrame 'df_gold_cliente' eliminado e deletado!
✅ DataFrame 'df_gold_produtos' eliminado e deletado!
✅ DataFrame 'df_gold_vendedores' eliminado e deletado!
✅ DataFrame 'df_silver_fato' eliminado e deletado!
✅ DataFrame 'df_fato_final' eliminado e deletado!
✅ DataFrame 'df_fato_contagem' eliminado e deletado!


In [0]:
# Definição da tabela
tabela = "fato"

# Caminhos das camadas Silver e Gold
gold_path_tabela = f"{gold_path}/{tabela}"
# Leitura da tabela Gold
df_gold = spark.read.format("delta").load(gold_path_tabela)
display(df_gold.count())
display(df_gold)
# Chame a função para eliminar todos os dataframes
limpeza_dataframes()

965446

VendasID,sk_produtos,sk_clientes,sk_vendedores,Quantidade,Desconto,PrecoTotal,DataVenda,PrecoUnitario,Ano,Mes,data_carga
1447191,155,48640,14,13,0.0,826.18,2018-02-07,63.5524,2018,2,2025-03-18T22:13:06.001+0000
233900,96,2782,21,1,0.2,62.0,2018-02-07,62.1993,2018,2,2025-03-18T22:13:06.001+0000
930189,105,15286,18,4,0.0,275.52,2018-02-07,68.8789,2018,2,2025-03-18T22:13:06.001+0000
4792802,205,26268,19,7,0.0,132.83,2018-02-07,18.9757,2018,2,2025-03-18T22:13:06.001+0000
6576409,338,65099,7,17,0.0,492.83,2018-02-07,28.9899,2018,2,2025-03-18T22:13:06.001+0000
5055748,301,62689,22,16,0.2,694.39,2018-02-07,43.412,2018,2,2025-03-18T22:13:06.001+0000
2331477,148,20064,19,6,0.0,171.33,2018-02-07,28.5553,2018,2,2025-03-18T22:13:06.001+0000
4670696,84,25450,15,7,0.0,632.11,2018-02-07,90.3019,2018,2,2025-03-18T22:13:06.001+0000
5864048,376,38020,3,10,0.0,408.13,2018-02-07,40.8126,2018,2,2025-03-18T22:13:06.001+0000
2989335,314,52652,3,14,0.0,78.42,2018-02-07,5.6015,2018,2,2025-03-18T22:13:06.001+0000


✅ DataFrame 'df_gold' eliminado e deletado!
